## **Task 3. Extrapolation**

**Points: 2**

**Forecast the 200 missing data points at the end of each series. Indicate the uncertainty of your estimate for each point!**
• Evaluate how well your proposed solutions are expected to work.
• Make sure to avoid overfitting and data leakage in your models.
• Can you use insights from Task 1 to improve extrapolation?
<br>
You are free to choose which methods to use for task 2 and 3. There are no rules other than the
obvious one that the methods must be statistically sound. It is probably best to use one method
for the interpolation problem (when data are missing within a series) and another method for the
extrapolation problem (when data is missing at the end of the series).


## Method: Block-wise PCA + VAR(2) Extrapolation

**Overview**

The 5 256 observed price observations are divided into **3 equal, non-overlapping time blocks**.  
Each block receives its own PCA decomposition and VAR(2) model — no data crosses block boundaries.  
The final 200-step forecast is produced by Block 3 (the most recent window).

**Why 3 blocks?**  
Financial price dynamics change over time (non-stationarity). Fitting separate models per block  
lets us (i) diagnose whether the factor structure shifts across time and (ii) ensure that the  
final forecast uses only the most recent dynamics.

**Data source & gap-filling**

`spiff_data-2.csv` is used as the base. The `1000`-sentinel values are replaced with NaN,  
then the file is merged with `spiff_data-2_interpolerad.csv` (Task 2 output) wherever that  
file provides non-NaN values.  
*Note:* the `_interpolated` columns in that file are a 1-step-lagged log-return reconstruction;  
they do not contain actual Bridge+GARCH fills for the NaN/sentinel gap positions.  
For the 55 positions per series (50-day block + 5 isolated sentinels) that remain NaN after  
the merge, Brownian Bridge geometric interpolation is applied — identical to the baseline  
method Task 2 uses (linear interpolation in log-price space = BB median path).

**Pipeline (strict no-leakage)**

| Step | What | Data used |
|------|------|-----------|
| 1 | Load raw + merge with Task-2 CSV | `spiff_data-2.csv` + interpolated file |
| 2 | Compute log-prices → log-returns | observed window only |
| 3 | Split 5 255 returns into 3 equal blocks (≈ 1 751 each) | no overlap |
| 4 | Standardise, fit PCA, fit VAR(2) | **block-local only** |
| 5 | Forecast 200 steps (Block 3 model), back-transform | — |
| 6 | Bootstrap 1 000 paths for prediction intervals | Block 3 residuals only |

**Hold-out validation** uses the last 200 return-steps of Block 3 as an unseen test set.  
PCA and VAR(2) are re-trained on Block 3 *minus* those 200 steps.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
np.random.seed(42)
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
BLOCK_COLORS = ['#2196F3', '#FF9800', '#4CAF50']   # blue / orange / green


In [ ]:
RAW_PATH    = '../data/spiff_data-2.csv'
INTERP_PATH = '../data/spiff_data-2_interpolerad.csv'

df_raw    = pd.read_csv(RAW_PATH,    index_col=0)
df_interp = pd.read_csv(INTERP_PATH, index_col=0)

PRICE_COLS  = ['gurkor', 'guitars', 'slingshots', 'stocks', 'sugar', 'water', 'tranquillity']
INTERP_COLS = [c + '_interpolated' for c in PRICE_COLS]

# ── Step 1: observed window from raw ──────────────────────────────────────────
last_obs_idx = int(df_raw[PRICE_COLS[0]].dropna().index[-1])   # 5255
prices_obs   = df_raw[PRICE_COLS].iloc[:last_obs_idx + 1].copy()
prices_obs.index = range(last_obs_idx + 1)

# ── Step 2: replace 1000-sentinel values with NaN ─────────────────────────────
prices_obs = prices_obs.replace(1000, np.nan)
print(f"NaN per series after 1000→NaN replacement : {prices_obs.isna().sum().iloc[0]} "
      f"(50 internal block + 5 isolated sentinels)")

# ── Step 3: merge with Task-2 interpolated CSV ────────────────────────────────
# interp_df row i+1 contains the reconstructed price for raw row i (1-step lag).
# For non-missing positions this matches raw exactly; for missing positions it is also
# NaN → the merge fills 0 gap positions but is done for correctness/transparency.
interp_aligned = df_interp[INTERP_COLS].iloc[1 : last_obs_idx + 2].copy()
interp_aligned.index  = range(last_obs_idx + 1)
interp_aligned.columns = PRICE_COLS

for col in PRICE_COLS:
    mask = prices_obs[col].isna()
    prices_obs.loc[mask, col] = interp_aligned.loc[mask, col]

n_remaining = prices_obs.isna().sum().iloc[0]
print(f"NaN remaining after merge with interpolated CSV : {n_remaining} "
      f"(file has no fills for these positions)")

# ── Step 4: Brownian Bridge geometric fill for remaining NaN ──────────────────
# BB median path = linear interpolation in log-price space = same baseline as Task 2.
def _fill_gaps_bb(s):
    """Fill all NaN gaps with BB geometric interpolation (linear in log-price)."""
    s = s.copy()
    nan_idx = s[s.isna()].index.tolist()
    if not nan_idx:
        return s
    gaps = []
    g0 = nan_idx[0]
    for i in range(1, len(nan_idx)):
        if nan_idx[i] != nan_idx[i - 1] + 1:
            gaps.append((g0, nan_idx[i - 1]))
            g0 = nan_idx[i]
    gaps.append((g0, nan_idx[-1]))

    for gs, ge in gaps:
        if gs == 0:                            # gap at start → forward fill
            s.iloc[gs:ge + 1] = s.dropna().iloc[0]
        elif ge == len(s) - 1:                 # gap at end → backward fill
            s.iloc[gs:ge + 1] = s.dropna().iloc[-1]
        else:                                  # interior gap → BB geometric
            p0, p1 = s.iloc[gs - 1], s.iloc[ge + 1]
            n  = ge - gs + 1
            t  = np.arange(1, n + 1)
            s.iloc[gs:ge + 1] = np.exp(np.log(p0) + t / (n + 1) * (np.log(p1) - np.log(p0)))
    return s

for col in PRICE_COLS:
    prices_obs[col] = _fill_gaps_bb(prices_obs[col])

assert prices_obs.isna().sum().sum() == 0, "Unexpected NaN after BB fill"
print(f"\nAll gaps filled.  Observed price matrix : {prices_obs.shape}")
print(f"Forecast horizon : {len(df_raw) - last_obs_idx - 1} steps (trailing NaN in raw)")


## Data Preparation

1. **Base data**: `spiff_data-2.csv`, observed window rows 0–5255 (5 256 prices per series).
2. **Sentinel replacement**: `1000` values → NaN (5 isolated positions per series).
3. **Merge**: NaN positions are matched against `spiff_data-2_interpolerad.csv`.  
   *The `_interpolated` columns there are a 1-step lagged reconstruction — they do not  
   contain fills for the gap positions, so the merge contributes 0 actual fills.*
4. **Brownian Bridge fill** for all remaining NaN (55 per series = 50-day contiguous block  
   + 5 isolated sentinels): geometric interpolation in log-price space, identical to Task 2's  
   BB baseline method (median path of a Brownian Bridge from p₀ to p₁ over the gap length).
5. **Log-prices → log-returns**: 5 255 stationary return vectors for PCA input.


In [ ]:
# ── Log-prices and log-returns ────────────────────────────────────────────────
log_prices = np.log(prices_obs)                   # (5256, 7)
log_returns = log_prices.diff().dropna()          # (5255, 7)
log_returns = log_returns.reset_index(drop=True)

print(f"Log-prices  : {log_prices.shape}")
print(f"Log-returns : {log_returns.shape}")
print(f"\nLog-return descriptives:\n{log_returns.describe().round(5)}")


## Block Split

The 5 255 log-returns are divided into **3 equal, non-overlapping blocks**.  
Each block is self-contained: standardisation, PCA and VAR(2) are fitted exclusively  
on that block's data.

| Block | Return indices | Approx. date range |
|-------|---------------|-------------------|
| 1     | 0 – B−1       | Days 1 – B        |
| 2     | B – 2B−1      | Days B+1 – 2B     |
| 3     | 2B – end      | Days 2B+1 – 5 256 |

The last price of each block serves as the back-transform anchor for that block.  
Block 3's last price (= last observed overall) anchors the final forecast.


In [ ]:
N_RET   = len(log_returns)          # 5255
B       = N_RET // 3                # 1751 – base block size
SIZES   = [B, B, N_RET - 2 * B]    # [1751, 1751, 1753]
STARTS  = [0, B, 2 * B]            # return-index start of each block

print(f"Total returns  : {N_RET}")
print(f"Block sizes    : {SIZES}  (sum={sum(SIZES)})")
print(f"Block starts   : {STARTS}")

# For each block we also need the log-price BEFORE the first return
# (anchor to back-transform forecast paths to price level).
# log_returns.iloc[i] = log_prices.iloc[i+1] - log_prices.iloc[i]
# → first return of block k is at return index STARTS[k]
#   → corresponds to log_prices.iloc[STARTS[k]+1] - log_prices.iloc[STARTS[k]]
#   → anchor = log_prices.iloc[STARTS[k]]
anchors = [log_prices.iloc[STARTS[k]] for k in range(3)]

blocks = []
for k in range(3):
    s, sz = STARTS[k], SIZES[k]
    blocks.append({
        'ret'    : log_returns.iloc[s : s + sz].copy().reset_index(drop=True),
        'anchor' : anchors[k],   # log-price vector at start of block
        'label'  : f'Block {k+1}',
        'color'  : BLOCK_COLORS[k],
    })
    print(f"  Block {k+1}: return indices [{s} .. {s+sz-1}], "
          f"price rows [{STARTS[k]} .. {STARTS[k]+sz}]")


## Per-Block PCA + VAR(2)

For each block:
1. **Standardise** log-returns (zero mean, unit variance) using *that block's* statistics only.
2. **Fit PCA** — keep `max(2, K)` components where K is the smallest number at which cumulative  
   explained variance exceeds 95 %. With Task 2 interpolated data the variance is spread across  
   components (PC1 ≈ 25–39 %), so 6–7 components are typically retained.
3. **Fit VAR(2)** on the PCA scores — explicit lag order 2.
4. Report eigenvalues, explained variance ratios and VAR diagnostics per block.


In [ ]:
K_MIN = 2    # minimum components for VAR to stay multivariate

fitted = []   # list of dicts, one per block

for k, blk in enumerate(blocks):
    ret = blk['ret'].values   # (size, 7)

    # ── Standardise ──────────────────────────────────────────────────────────
    sc  = StandardScaler()
    std = sc.fit_transform(ret)

    # ── PCA ──────────────────────────────────────────────────────────────────
    pca = PCA()
    pca.fit(std)
    evr    = pca.explained_variance_ratio_
    cumevr = np.cumsum(evr)
    K      = max(K_MIN, int(np.searchsorted(cumevr, 0.95)) + 1)

    scores    = pca.transform(std)[:, :K]
    scores_df = pd.DataFrame(scores, columns=[f'PC{i+1}' for i in range(K)])

    # ── VAR(2) ───────────────────────────────────────────────────────────────
    var_res = VAR(scores_df).fit(2)

    fitted.append({
        'label'    : blk['label'],
        'color'    : blk['color'],
        'scaler'   : sc,
        'pca'      : pca,
        'K'        : K,
        'evr'      : evr,
        'cumevr'   : cumevr,
        'scores_df': scores_df,
        'var'      : var_res,
        'anchor'   : blk['anchor'],
    })

    print(f"── {blk['label']} ──────────────────────────────────────────────")
    print(f"  Eigenvalues  : {np.round(pca.explained_variance_, 4)}")
    print(f"  EVR          : {np.round(evr, 4)}")
    print(f"  Cum. EVR     : {np.round(cumevr, 4)}")
    print(f"  K kept       : {K}")
    print(f"  VAR(2) AIC   : {var_res.aic:.4f}   BIC: {var_res.bic:.4f}\n")


In [ ]:
# ── Scree plots (all 3 blocks side by side) ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for k, f in enumerate(fitted):
    ax = axes[k]
    n  = len(f['evr'])
    ax.bar(range(1, n+1), f['evr'] * 100, color=f['color'], alpha=0.8)
    ax2 = ax.twinx()
    ax2.plot(range(1, n+1), f['cumevr'] * 100, 'k--o', markersize=4)
    ax2.axhline(95, color='red', lw=0.8, ls='--')
    ax.set_title(f["label"])
    ax.set_xlabel('Component')
    ax.set_ylabel('EVR (%)')
    ax2.set_ylabel('Cum. EVR (%)')
    ax2.set_ylim(0, 105)
fig.suptitle('Scree plots by block', fontsize=13)
plt.tight_layout()
plt.show()

# ── PC1 loadings comparison ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(PRICE_COLS))
w = 0.25
for k, f in enumerate(fitted):
    # PC1 loading = first row of components_ (or first column of V in SVD)
    loadings = np.abs(f['pca'].components_[0])
    ax.bar(x + (k - 1) * w, loadings, width=w, label=f["label"], color=f['color'], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(PRICE_COLS, rotation=20)
ax.set_ylabel('|PC1 loading|')
ax.set_title('PC1 absolute loadings per block  (factor structure stability)')
ax.legend()
plt.tight_layout()
plt.show()


## Hold-out Validation (Block 3 only)

The last 200 return-steps of Block 3 are held out as an unseen test set.  
PCA and VAR(2) are re-trained on Block 3 *minus* those 200 steps, following the same  
block-local standardisation — no data from Blocks 1 or 2 is used.

Metrics: RMSE, MAE, MAPE per series.


In [ ]:
HOLDOUT = 200
blk3    = blocks[2]
ret3    = blk3['ret']   # (1753, 7)

# ── Split Block 3 ─────────────────────────────────────────────────────────────
train3  = ret3.iloc[: -HOLDOUT].copy()   # (1553, 7)
hold3   = ret3.iloc[-HOLDOUT :].copy()   # (200,  7)

# Anchor for back-transform = log-price before the HOLD-OUT period starts
# The hold-out starts at return index -200 within Block 3.
# Returns in Block 3 start at price index STARTS[2] in log_prices.
# Hold-out start in log_prices = STARTS[2] + len(train3)
anchor_ho = log_prices.iloc[STARTS[2] + len(train3)]

print(f"Block 3 total returns : {len(ret3)}")
print(f"Hold-out training     : {train3.shape}")
print(f"Hold-out test         : {hold3.shape}")

# ── PCA + VAR(2) on Block 3 training set ──────────────────────────────────────
sc_ho   = StandardScaler()
std_ho  = sc_ho.fit_transform(train3.values)

pca_ho  = PCA()
pca_ho.fit(std_ho)
evr_ho  = pca_ho.explained_variance_ratio_
K_ho    = max(K_MIN, int(np.searchsorted(np.cumsum(evr_ho), 0.95)) + 1)

sc_df_ho = pd.DataFrame(pca_ho.transform(std_ho)[:, :K_ho],
                         columns=[f'PC{i+1}' for i in range(K_ho)])
var_ho   = VAR(sc_df_ho).fit(2)

print(f"\nHold-out model  K={K_ho}  AIC={var_ho.aic:.4f}  BIC={var_ho.bic:.4f}")

# ── Forecast 200 steps ────────────────────────────────────────────────────────
def back_transform(paths, pca_model, scaler, K, last_lp):
    """
    paths    : (n_paths, steps, K)
    Returns  : (n_paths, steps, 7) prices
    """
    n, s, _ = paths.shape
    full = np.zeros((n, s, pca_model.n_components_))
    full[:, :, :K] = paths
    std_ret = full @ pca_model.components_          # (n, s, 7)
    log_ret = std_ret * scaler.scale_ + scaler.mean_
    log_p   = np.cumsum(log_ret, axis=1) + last_lp.values
    return np.exp(log_p)

init_ho    = sc_df_ho.values[-var_ho.k_ar:]
fc_scores  = var_ho.forecast(init_ho, steps=HOLDOUT)           # (200, K_ho)
fc_prices_ho = back_transform(fc_scores[np.newaxis], pca_ho,
                               sc_ho, K_ho, anchor_ho)[0]       # (200, 7)

# True hold-out prices
true_lp_ho  = np.cumsum(hold3.values, axis=0) + anchor_ho.values
true_prices_ho = np.exp(true_lp_ho)

# ── Metrics ───────────────────────────────────────────────────────────────────
rows_m = []
for j, col in enumerate(PRICE_COLS):
    p, t = fc_prices_ho[:, j], true_prices_ho[:, j]
    rows_m.append({'series': col,
                   'RMSE'    : np.sqrt(mean_squared_error(t, p)),
                   'MAE'     : mean_absolute_error(t, p),
                   'MAPE (%)': np.mean(np.abs((t - p) / t)) * 100})
metrics_df = pd.DataFrame(rows_m).set_index('series')
print("\nHold-out evaluation (last 200 returns of Block 3):\n")
print(metrics_df.round(4))


In [ ]:
# ── Hold-out plots ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.ravel()
for j, col in enumerate(PRICE_COLS):
    ax = axes[j]
    ax.plot(true_prices_ho[:, j], color='black',     lw=1.5, label='True')
    ax.plot(fc_prices_ho[:,  j],  color='steelblue', lw=1.5, ls='--', label='Forecast')
    ax.set_title(f'{col}   RMSE={metrics_df.loc[col,"RMSE"]:.3f}'
                 f'   MAPE={metrics_df.loc[col,"MAPE (%)"]:.2f}%')
    ax.set_xlabel('Hold-out step')
    ax.set_ylabel('Price')
    ax.legend()
axes[-1].axis('off')
fig.suptitle('Hold-out Validation (Block 3): True vs PCA+VAR(2) Forecast', fontsize=14)
plt.tight_layout()
plt.show()


## Final Forecast: 200 Missing Steps  (Block 3 model)

PCA and VAR(2) are re-trained on **all of Block 3** (1 753 return steps).  
This is entirely disjoint from Blocks 1 and 2. The forecast horizon is 200 steps beyond  
the last observed price.

**Uncertainty — residual bootstrap:**  
1 000 paths are simulated by iterating the VAR(2) coefficient matrices and adding  
block-local VAR residuals sampled with replacement at each step.  
Each path is back-transformed to price level, giving 80 % and 95 % prediction intervals.


In [ ]:
# ── PCA + VAR(2) on full Block 3 ─────────────────────────────────────────────
f3      = fitted[2]     # already computed in per-block loop
sc3     = f3['scaler']
pca3    = f3['pca']
K3      = f3['K']
var3    = f3['var']

# Last log-price anchor (very last observed price)
last_lp = log_prices.iloc[-1]

print(f"Block 3  K={K3}  AIC={var3.aic:.4f}  BIC={var3.bic:.4f}")
print(f"Last observed log-price:\n{last_lp.round(4)}")

# ── Point forecast ────────────────────────────────────────────────────────────
init3    = f3['scores_df'].values[-var3.k_ar:]
fc_det   = var3.forecast(init3, steps=200)      # (200, K3)
fc_final = back_transform(fc_det[np.newaxis], pca3, sc3, K3, last_lp)[0]   # (200, 7)

# ── Bootstrap ─────────────────────────────────────────────────────────────────
N_SIM   = 1000
LAG     = var3.k_ar
coefs   = var3.coefs
intcpt  = var3.intercept
resids  = np.asarray(var3.resid)

sim_paths = np.zeros((N_SIM, 200, K3))
for s in range(N_SIM):
    eps  = resids[np.random.choice(len(resids), size=200, replace=True)]
    hist = list(init3)
    for t in range(200):
        z = intcpt.copy()
        for lag_i in range(LAG):
            z += coefs[lag_i] @ hist[-1 - lag_i]
        z += eps[t]
        hist.append(z)
    sim_paths[s] = np.array(hist[LAG:])

sim_prices = back_transform(sim_paths, pca3, sc3, K3, last_lp)   # (1000, 200, 7)

PI = {
    'lower_95': np.percentile(sim_prices,  2.5, axis=0),
    'upper_95': np.percentile(sim_prices, 97.5, axis=0),
    'lower_80': np.percentile(sim_prices, 10.0, axis=0),
    'upper_80': np.percentile(sim_prices, 90.0, axis=0),
}
print(f"\nBootstrap done: {N_SIM} paths × 200 steps × 7 series")
print("\nPoint forecast – first 5 steps:")
print(pd.DataFrame(fc_final[:5], columns=PRICE_COLS).round(4))


In [ ]:
# ── Final forecast plots ──────────────────────────────────────────────────────
CONTEXT = 100
steps   = np.arange(1, 201)

fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.ravel()

for j, col in enumerate(PRICE_COLS):
    ax = axes[j]
    hist_x = np.arange(-CONTEXT, 0)
    ax.plot(hist_x, prices_obs[col].values[-CONTEXT:],
            color='black', lw=1.2, label='Observed')
    ax.fill_between(steps, PI['lower_95'][:, j], PI['upper_95'][:, j],
                    alpha=0.15, color=BLOCK_COLORS[2], label='95% PI')
    ax.fill_between(steps, PI['lower_80'][:, j], PI['upper_80'][:, j],
                    alpha=0.32, color=BLOCK_COLORS[2], label='80% PI')
    ax.plot(steps, fc_final[:, j],
            color=BLOCK_COLORS[2], lw=1.5, ls='--', label='Point forecast (Block 3)')
    ax.axvline(x=0.5, color='gray', ls=':', lw=1)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Step  (0 = last observed,  1–200 = forecast)')
    ax.set_ylabel('Price')
    ax.legend(fontsize=8)

axes[-1].axis('off')
fig.suptitle('Final Forecast: Block 3 PCA+VAR(2) with Bootstrap Prediction Intervals',
             fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ── Build forecast_task3 DataFrame ───────────────────────────────────────────
records = []
for step_i in range(200):
    for j, col in enumerate(PRICE_COLS):
        records.append({
            'step'          : step_i + 1,
            'series'        : col,
            'forecast_price': fc_final[step_i, j],
            'lower_80'      : PI['lower_80'][step_i, j],
            'upper_80'      : PI['upper_80'][step_i, j],
            'lower_95'      : PI['lower_95'][step_i, j],
            'upper_95'      : PI['upper_95'][step_i, j],
        })

forecast_task3 = pd.DataFrame(records)
print(f"forecast_task3  shape : {forecast_task3.shape}")
print(f"\nFirst 14 rows:\n{forecast_task3.head(14).to_string(index=False)}")
print(f"\nLast 7 rows:\n{forecast_task3.tail(7).to_string(index=False)}")
